# Contact: pressing a sphere into a soft slab

**The question.** Real soft-body work is full of contact. When a rigid sphere presses into a soft slab, what does each solver actually give you?

**The test.** Push a rigid sphere into a clamped slab in small steps. FEM resolves the contact with a penalty / Augmented-Lagrangian law written in pure UFL (no C++); Newton enforces non-penetration through its `soft_contact` pipeline.

**The three Newton solvers** (NVIDIA Newton is a GPU soft-body engine built on Warp/CUDA — same physics, different numerics):

* **XPBD** — *Extended Position-Based Dynamics*: fast positional projection (the game/robotics default). It satisfies constraints by moving positions, never solving a force balance, so it leaves a finite residual and reads soft.
* **VBD** — *Vertex Block Descent*: implicit; block coordinate descent on the backward-Euler objective over a coloured vertex graph. Converges toward the FEM-like solution as iterations grow — the apples-to-apples match for the implicit FEM.
* **SemiImplicit** — explicit, force-based (symplectic / semi-implicit Euler: a velocity-then-position step, no implicit solve). The one Warp can differentiate, used for the θ\* stiffness fit.

**The verdict.** FEM produces a **calibrated contact-force curve** (anchored against the analytic Hertz solution); **XPBD exposes no comparable force** — it only gives deformation. So the honest comparison axis is the **deformed shape**, and FEM's force curve is something the fast positional solver cannot provide. The FEM variants form a gradation from coarse to accurate contact: `tet kn×5 penalty` → `tet kn×50 penalty` → `tet kn×50 AL` → `hex kn×50 AL` (penalty vs AL at equal kn, then tet vs locking-free hex; AL → penetration ≈ 0).

Newton now runs **all three** solvers on this same contact scene (`--solver xpbd|vbd|semi_implicit`, default XPBD); the implicit **VBD** is the apples-to-apples match for the implicit FEM. Whichever solver runs are present on disk are overlaid below — the VBD/explicit contact path needs a recent Newton build, so on an older version only XPBD may be present.

In [ ]:
%matplotlib inline
import os

import matplotlib.pyplot as plt
import numpy as np

from common import params
from compare import indentation as cmp_ind   # shared make_* helpers (single source for the dimple)
from compare import style

if not os.path.exists(params.FEM_INDENT_NPZ):
    raise FileNotFoundError(
        'data/fem_indentation.npz missing -- run `python -m fenics_run.run_indentation` first')
fem = np.load(params.FEM_INDENT_NPZ)
# all present Newton runs, in canonical order / colour / marker
newtons = style.load_newton_runs(params.NEWTON_INDENT_NPZ)
newton_label, newton = (newtons[0][0], newtons[0][1]) if newtons else (None, None)
slugs = [k[3:] for k in fem.files if k.startswith('uz_')]
d = fem['deltas'] * 1000.0
print('FEM variants:', slugs, '| Newton runs:', [lbl for lbl, *_ in newtons] or 'none')

## The scene — what we're actually simulating

A clamped soft slab (1.0 × 1.0 × 0.4 m) with a rigid sphere (R = 0.30 m) lowered onto its top face in small steps. Below is Newton's slab at maximum indentation — rendered from the **real mesh** once `run_indentation` has run (a setup schematic otherwise) — with the sphere as a wireframe. The sphere is treated **analytically** (its gap to the deformed surface is closed-form), so there is no mesh-to-mesh contact search; that is what keeps the whole contact law inside UFL.

In [ ]:
from compare import scene

fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111, projection="3d")
if newton is not None and "tet_indices" in newton.files:
    norm, lab = scene.render(ax, newton["rest_q"], newton["final_q"], newton["tet_indices"],
                             ghost_rest=False, title=f"{newton_label} - indented slab (real mesh)")
    scene.add_sphere(ax, newton["sphere_c"], float(newton["sphere_r"]))
    scene.add_colorbar(fig, ax, norm, lab)
else:
    nx, ny, nz = params.INDENT_DIM
    h = params.INDENT_CELL
    R = params.INDENT_SPHERE_R
    Lx, Ly, Lz = nx * h, ny * h, nz * h
    scene.draw_box(ax, (0, 0, 0), (Lx, Ly, Lz), frame_pts=[[0, 0, 0], [Lx, Ly, Lz + R + 0.05]])
    scene.add_sphere(ax, (Lx / 2.0, Ly / 2.0, Lz + R - params.INDENT_MAX), R)
    ax.set_title("Setup schematic (run newton_run.run_indentation for the real dimple)")
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_zlabel("z [m]")
ax.view_init(elev=18, azim=-60)
plt.tight_layout(); plt.show()

## 1. The verdict — a contact-force curve only FEM gives here

As the sphere presses in, FEM assembles the **total contact force** at each depth; the dashed line is the analytic **Hertz** solution for a rigid sphere on an elastic half-space (an approximate anchor — our slab is finite and soft). The Newton runs report deformation (dimple, penetration, strain energy) but **no calibrated contact force**, so this axis is necessarily FEM-only — and that absence *is* the point. (It reflects the fast positional contact handling, not a fundamental limit; see `docs/CONTACT.md`.)

In [ ]:
plt.figure(figsize=(6, 5))
for i, s in enumerate(slugs):
    plt.plot(d, fem["f_" + s], "o-", color=style.fem_variant_color(i), label="FEM " + s)
plt.plot(d, fem["f_hertz"], color=style.COLOR["analytic"], ls=style.ANALYTIC_LS, lw=1.5,
         label="Hertz (analytic anchor)")
plt.xlabel("indentation depth [mm]"); plt.ylabel("contact force [N]")
plt.legend(); plt.grid(alpha=0.3)
plt.title("Indentation - contact force (Newton exposes none)"); plt.show()

## 2. The honest comparison — the deformed dimple

Since only deformation is common to both solvers, this is where they meet: the vertical displacement of the top surface along a line through the contact centre. FEM variants and whichever Newton solver runs are present are overlaid — the same physical dimple, seen by the fast positional solver and by implicit FEM.

In [ ]:
# Single-source figure: the same make_dimple the pipeline saves to figures/ (compare/indentation.py).
cmp_ind.make_dimple(fem, newtons)
plt.show()

## 3. Residual penetration — how far the body sinks in (all solvers)

How much each solver lets the body penetrate the obstacle, with **every solver overlaid** — the cleanest "coarse → accurate" contact axis. On the FEM side, plain **penalty** leaves a penetration set by the stiffness `kn`, while the **Augmented-Lagrangian** Uzawa loop drives it toward zero at the *same* modest `kn` (approaching exact non-penetration): `tet kn×5 penalty` (most) collapses to the `AL` variants (≈ 0). Newton's positional `soft_contact` penetration is plotted alongside, so it sits directly against the FEM gradation.

In [ ]:
plt.figure(figsize=(6, 5))
for i, s in enumerate(slugs):
    if ("pen_" + s) in fem.files:
        plt.plot(d, fem["pen_" + s] * 1000.0, "o-", color=style.fem_variant_color(i), label="FEM " + s)
for label, nd, color, marker in newtons:
    plt.plot(nd["deltas"] * 1000.0, nd["penetration"] * 1000.0, color=color, marker=marker, ms=3, label=label)
plt.xlabel("indentation depth [mm]"); plt.ylabel("max residual penetration [mm]")
plt.legend(); plt.grid(alpha=0.3)
plt.title("Indentation - residual penetration (FEM AL -> ~0 vs Newton positional)"); plt.show()

## 4. Internal (strain) energy vs indentation

How much elastic energy the slab stores as the sphere presses in — comparable across variants and Newton because it uses the same Neo-Hookean density on the same kind of mesh.

In [ ]:
plt.figure(figsize=(6, 5))
for i, s in enumerate(slugs):
    plt.plot(d, fem["e_strain_" + s], "o-", color=style.fem_variant_color(i), label="FEM " + s)
for label, nd, color, marker in newtons:
    plt.plot(nd["deltas"] * 1000.0, nd["e_strain"], color=color, marker=marker, ms=3, label=label)
plt.xlabel("indentation depth [mm]"); plt.ylabel("strain energy [J]")
plt.legend(); plt.grid(alpha=0.3)
plt.title("Indentation - internal energy"); plt.show()

## 5. Contact (penalty) energy — FEM only

The penalty potential `½ kn <-g>+²` stored in the FEM contact layer. A stiffer `kn` stores less of it (less penetration); the Augmented-Lagrangian variant keeps it small even at modest `kn`. This is a property of the FEM penalty formulation — the Newton runs report no comparable stored contact energy, so this axis is necessarily FEM-only.

In [ ]:
plt.figure(figsize=(6, 5))
for i, s in enumerate(slugs):
    plt.plot(d, fem["e_contact_" + s], "o-", color=style.fem_variant_color(i), label="FEM " + s)
plt.xlabel("indentation depth [mm]"); plt.ylabel("contact (penalty) energy [J]")
plt.legend(); plt.grid(alpha=0.3)
plt.title("Indentation - contact energy (FEM penalty)"); plt.show()

## 6. Did the Newton solvers settle? — a Newton-only diagnostic

Newton is a dynamic solver, so before reading its "settled" state at each indentation step we confirm the residual kinetic energy fell to ≈ 0 — overlaid for every present Newton solver. FEM is static and has no such transient, so this panel is necessarily Newton-only (the penetration itself is compared against FEM in §3).

In [ ]:
if newtons:
    plt.figure(figsize=(6, 4))
    for label, nd, color, marker in newtons:
        plt.semilogy(nd["deltas"] * 1000.0, np.maximum(nd["ke"], 1e-12),
                     color=color, marker=marker, ms=3, label=label)
    plt.xlabel("indentation depth [mm]"); plt.ylabel("residual KE [J]  (log scale)")
    plt.title("Indentation - did each Newton solver settle? (KE -> ~0)")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
else:
    print("No Newton indentation result yet -- run: "
          "python -m newton_run.run_indentation [--solver vbd|semi_implicit]")

## 7. Computation time (wall-clock)

The cost of each variant (and Newton), per the same trade-off as the hanging bar: the heavier the contact model (more penalty iterations, Augmented-Lagrangian outer loops), the more time it takes.

In [ ]:
ft, fc = {}, []
for i, s in enumerate(slugs):
    if ("time_" + s) in fem.files:
        ft[s] = float(fem["time_" + s]); fc.append(style.fem_variant_color(i))
for label, nd, color, marker in newtons:
    if "wall_time" in nd.files:
        ft[label] = float(nd['wall_time']); fc.append(color)
for k, v in ft.items():
    print(f"{k:16s}: {v:8.3f} s")
plt.figure(figsize=(6, 4))
plt.bar(list(ft), list(ft.values()), color=fc)
plt.ylabel("wall time [s]"); plt.title("Indentation - computation time"); plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

## Takeaway

* FEM gives a **calibrated contact-force curve** that follows the Hertz `δ^{3/2}` trend and deviates as the contact radius grows relative to the finite slab — a curve the **Newton runs do not produce** (§1).
* Across the FEM gradation, **stiffer penalty → less penetration**, and the **Augmented Lagrangian drives penetration ≈ 0** at modest `kn` (§3); on locking-free hex it is the most accurate of the four. Newton's positional penetration sits in that same plot.
* So the solvers are honestly compared wherever they all report data — the deformed **dimple** (§2), **residual penetration** (§3), **strain energy** (§4) and **cost** (§7); the contact **force** (§1) and penalty **energy** (§5) are FEM-only by construction, and the **settle check** (§6) is Newton-only.
* Friction is deliberately left out here (frictionless keeps the Hertz comparison clean); it is its own scenario in `40_friction.ipynb`.